# 11 제품 수요패턴 · RIDR 분석

논문 3장 스타일의 **제품(시계열) 수요패턴 분석**을 수행합니다.

- **System-Level / SKU-Level** 기초통계 (type별, Table 3.2 스타일)
- **SBC / ML 클러스터**별 System·SKU-Level 변동성 요약
- **RIDR** (Relative Inter-Demand Range): `(P90 − P10) / median`
- 클러스터별 **10–90% 수요 밴드** 시각화 (type 간 비교)

> Ecuador 데이터: 논문의 센터 A/B → 매장 **type** (D, B, C, E, A)으로 대응  
> 분석 단위: **type × family** (165 주간 시계열)

**선행 노트북**: 05(SBC), 06 또는 09(ML 클러스터) 실행 후 진행

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED, SBC_CLUSTER, ML_CLUSTER
from utils.stats_summary import system_level_weekly_stats, sku_level_weekly_stats
from utils.pattern_analysis import (
    merge_cluster_labels,
    compute_system_level_cluster_stats,
    compute_sku_level_cluster_summary,
    compute_ridr_summary_by_type,
    band_percentile_summary,
    plot_cluster_bands_by_type,
)
from utils.sbc import CLUSTER_LABELS

dfw = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')
sbc = pd.read_parquet(SBC_CLUSTER)
ml = pd.read_parquet(ML_CLUSTER) if ML_CLUSTER.exists() else None
df = merge_cluster_labels(dfw, sbc, ml)

types = sorted(dfw['type'].unique())
COMPARE_TYPES = [types[0], types[-1]]  # 예: D vs A (논문 Center A vs B 대응)
print('types:', types, '| compare:', COMPARE_TYPES)

## 1. System-Level / SKU-Level 기초통계 (type별)

In [ ]:
system_rows, sku_rows, sku_detail_all = [], [], []
for t in types:
    system_rows.append(system_level_weekly_stats(dfw, t))
    sku_sum, sku_detail = sku_level_weekly_stats(dfw, t)
    sku_rows.append(sku_sum)
    sku_detail_all.append(sku_detail)

system_stats = pd.concat(system_rows, ignore_index=True)
sku_stats = pd.concat(sku_rows, ignore_index=True)
sku_detail = pd.concat(sku_detail_all, ignore_index=True)

print('=== System-Level (type별 주간 총판매) ===')
display(system_stats.round(3))
print('\n=== SKU-Level 요약 (type별 family 시계열) ===')
display(sku_stats.round(3))

## 2. SBC 클러스터별 System / SKU-Level 변동성 (type별)

In [ ]:
sbc_sys_all, sbc_sku_all = [], []
for t in types:
    sub = df[df['type'] == t]
    _, sys_st = compute_system_level_cluster_stats(sub, 'SBC_CLUSTER')
    _, sku_st = compute_sku_level_cluster_summary(sub, 'SBC_CLUSTER')
    sys_st['type'] = t
    sku_st['type'] = t
    sbc_sys_all.append(sys_st)
    sbc_sku_all.append(sku_st)
    label = CLUSTER_LABELS.get(1, '')
    print(f'\n--- Type {t}: SBC System-Level ---')
    display(sys_st.round(3))
    print(f'--- Type {t}: SBC SKU-Level ---')
    display(sku_st.round(3))

sbc_system_by_type = pd.concat(sbc_sys_all, ignore_index=True)
sbc_sku_by_type = pd.concat(sbc_sku_all, ignore_index=True)

## 3. ML 클러스터별 System / SKU-Level 변동성 (type별)

In [ ]:
ml_sys_all, ml_sku_all = [], []
if 'ML_CLUSTER' in df.columns:
    for t in types:
        sub = df[df['type'] == t]
        _, sys_st = compute_system_level_cluster_stats(sub, 'ML_CLUSTER')
        _, sku_st = compute_sku_level_cluster_summary(sub, 'ML_CLUSTER')
        sys_st['type'] = t
        sku_st['type'] = t
        ml_sys_all.append(sys_st)
        ml_sku_all.append(sku_st)
        print(f'\n--- Type {t}: ML System-Level ---')
        display(sys_st.round(3))
        print(f'--- Type {t}: ML SKU-Level ---')
        display(sku_st.round(3))
    ml_system_by_type = pd.concat(ml_sys_all, ignore_index=True)
    ml_sku_by_type = pd.concat(ml_sku_all, ignore_index=True)
else:
    print('ML_CLUSTER 없음 — 06 또는 09 노트북을 먼저 실행하세요.')

## 4. RIDR 지수 · 클러스터 수요 밴드 (SBC)

RIDR = `(P90 − P10) / median` — 클러스터 내 시계열 수요 분산의 상대적 폭

In [ ]:
sbc_bands, sbc_ridr = compute_ridr_summary_by_type(df, 'SBC_CLUSTER')
print('=== SBC: RIDR Summary by Cluster (type별) ===')
print(sbc_ridr.round(3))
print('\n=== SBC: 10th/90th Percentile Band (mean over weeks) ===')
print(band_percentile_summary(sbc_bands, 'SBC_CLUSTER', 'type'))

fig = plot_cluster_bands_by_type(
    sbc_bands, 'SBC_CLUSTER', types_to_plot=COMPARE_TYPES, method_label='SBC'
)
plt.show()

## 5. RIDR 지수 · 클러스터 수요 밴드 (ML)

In [ ]:
if 'ML_CLUSTER' in df.columns:
    ml_bands, ml_ridr = compute_ridr_summary_by_type(df, 'ML_CLUSTER')
    print('=== ML: RIDR Summary by Cluster (type별) ===')
    print(ml_ridr.round(3))
    print('\n=== ML: 10th/90th Percentile Band (mean over weeks) ===')
    print(band_percentile_summary(ml_bands, 'ML_CLUSTER', 'type'))
    fig = plot_cluster_bands_by_type(
        ml_bands, 'ML_CLUSTER', types_to_plot=COMPARE_TYPES, method_label='ML'
    )
    plt.show()

In [ ]:
out = DATA_PROCESSED
system_stats.to_csv(out / 'type_system_level_stats.csv', index=False)
sku_stats.to_csv(out / 'type_sku_level_stats.csv', index=False)
sku_detail.to_csv(out / 'type_family_series_stats.csv', index=False)
sbc_system_by_type.to_csv(out / 'sbc_system_level_by_type.csv', index=False)
sbc_sku_by_type.to_csv(out / 'sbc_sku_level_by_type.csv', index=False)
sbc_ridr.to_csv(out / 'sbc_ridr_by_type.csv')
sbc_bands.to_parquet(out / 'sbc_cluster_bands.parquet', index=False)

if 'ML_CLUSTER' in df.columns:
    ml_system_by_type.to_csv(out / 'ml_system_level_by_type.csv', index=False)
    ml_sku_by_type.to_csv(out / 'ml_sku_level_by_type.csv', index=False)
    ml_ridr.to_csv(out / 'ml_ridr_by_type.csv')
    ml_bands.to_parquet(out / 'ml_cluster_bands.parquet', index=False)

print('저장 완료:', out)

## 종합 분석 (논문 3장 스타일)

### 1. System-Level / SKU-Level 기초통계
- **type D·A** 주간 총판매 규모 최대(~145만), **type E** 최소(~24.7만)
- System-Level CV: E(0.43) > B(0.34) > D·A(0.30) — 소규모 type일수록 상대 변동성 큼
- SKU-Level 평균 CV: E(0.87) > C(0.72) > D(0.62) — **type D가 family 단위에서 가장 안정적**

### 2. SBC 클러스터별 RIDR 해석
| Cluster | 해석 | RIDR 특징 |
|---------|------|-----------|
| **1 Smooth** | 연속·안정 수요 | type 간 RIDR 7~9로 **균일** — 패턴이 type에 관계없이 유사 |
| **2 Intermittent** | 간헐·저변동 | type D(5.4) vs B·E(47+) — **type 간 격차 극대** |
| **3 Erratic** | 연속·고변동 | type C에서 RIDR≈0 (시계열 수 적음) |
| **4 Lumpy** | 간헐·고변동 | type A(250.8) vs D(1.3) — **A type 루미 수요의 이질성 매우 큼** |

→ 논문의 센터 A/B 비교와 같이, Ecuador에서도 **동일 SBC 클러스터 내에서 type별 수요 분산 구조가 크게 다름**

### 3. ML 클러스터 RIDR
- ML Cluster 1: type E RIDR=107.7로 **간헐·이질 수요 집중**
- ML Cluster 4: type 간 RIDR 2.7~8.0으로 **비교적 균일** (Smooth에 가까운 군집)
- SBC 대비 ML은 **데이터 기반 재분류**로 type E의 고-RIDR 시계열을 별도 클러스터로 분리

### 4. 시사점
- **RIDR이 높은 클러스터**(SBC-4 type A, SBC-2 type B/E)는 클러스터 내 시계열 간 수요 패턴 이질성이 크므로 **군집별 차별 예측 모델** 필요
- **RIDR이 낮은 클러스터**(SBC-1 전 type, ML-4)는 단일 모델로도 묶어 예측 가능
- 10장 하이브리드 프레임워크에서 SBC vs ML 클러스터링 효과 검증의 근거 자료